In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Warehouse Utilization

## Difficulty
Medium

## Topics
- PySpark
- SQL
- Joins
- Aggregation
- Group By
- Arithmetic Operations
- Sorting

## Problem Statement

You are given two PySpark DataFrames:

### Dataset 1: `warehouse`

| Column | Data Type | Description |
|---|---|---|
| `name` | String | Name of the warehouse |
| `product_id` | Integer | Identifier of the product stored |
| `units` | Integer | Number of units of the product stored |

### Dataset 2: `products`

| Column | Data Type | Description |
|---|---|---|
| `product_id` | Integer | Unique product identifier |
| `product_name` | String | Name of the product |
| `width` | Integer | Width of one unit in centimeters |
| `length` | Integer | Length of one unit in centimeters |
| `height` | Integer | Height of one unit in centimeters |

## Task

Calculate the **total volume occupied by products in each warehouse**.

The volume occupied by each product entry is calculated as:

`width × length × height × units`

You must:

1. Join the `warehouse` and `products` DataFrames using `product_id`.
2. Calculate the volume occupied by each product entry.
3. Group the data by warehouse.
4. Calculate the total volume for each warehouse.
5. Return the following columns:
   - `warehouse_name`
   - `total_volume`
6. Sort the result by `total_volume` in **descending order**.

## Expected Output

| warehouse_name | total_volume |
|---|---:|
| Central Hub | 4200000 |
| North Depot | 2448000 |
| South Wing | 1800000 |

In [2]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

warehouse_data = [
    ("Central Hub", 1, 100),
    ("Central Hub", 2, 50),
    ("North Depot", 1, 200),
    ("North Depot", 3, 80),
    ("South Wing", 2, 30)
]

warehouse_schema = StructType([
    StructField("name", StringType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("units", IntegerType(), False)
])

warehouse = spark.createDataFrame(
    warehouse_data,
    warehouse_schema
)

warehouse.show()
warehouse.printSchema()

+-----------+----------+-----+
|       name|product_id|units|
+-----------+----------+-----+
|Central Hub|         1|  100|
|Central Hub|         2|   50|
|North Depot|         1|  200|
|North Depot|         3|   80|
| South Wing|         2|   30|
+-----------+----------+-----+

root
 |-- name: string (nullable = false)
 |-- product_id: integer (nullable = false)
 |-- units: integer (nullable = false)



In [3]:
products_data = [
    (1, "Laptop Box", 40, 30, 10),
    (2, "Monitor Box", 60, 50, 20),
    (3, "Phone Box", 15, 8, 5)
]

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("width", IntegerType(), False),
    StructField("length", IntegerType(), False),
    StructField("height", IntegerType(), False)
])

products = spark.createDataFrame(
    products_data,
    products_schema
)

products.show()
products.printSchema()

+----------+------------+-----+------+------+
|product_id|product_name|width|length|height|
+----------+------------+-----+------+------+
|         1|  Laptop Box|   40|    30|    10|
|         2| Monitor Box|   60|    50|    20|
|         3|   Phone Box|   15|     8|     5|
+----------+------------+-----+------+------+

root
 |-- product_id: integer (nullable = false)
 |-- product_name: string (nullable = false)
 |-- width: integer (nullable = false)
 |-- length: integer (nullable = false)
 |-- height: integer (nullable = false)



In [4]:
products.createOrReplaceTempView("products")
warehouse.createOrReplaceTempView("warehouse")

In [13]:
spark.sql("""
        with CTE as (
                    SELECT *,
                        width*length*height*units AS volume 
                    from 
                    products p join warehouse w 
                    on p.product_id = w.product_id
                    )
           SELECT 
               name as warehouse_name
               ,sum(volume) as total_volume 
            from cte 
            group by name 
            order by total_volume DESC
""").show()

+--------------+------------+
|warehouse_name|total_volume|
+--------------+------------+
|   Central Hub|     4200000|
|   North Depot|     2448000|
|    South Wing|     1800000|
+--------------+------------+



In [27]:
from pyspark.sql.functions import *
products\
    .join(
        warehouse, 
        on = "product_id", 
        how = "inner"
        )\
        .withColumn(
            "volume",
            col("width")*col("height")*col("length")*col("units")
        )\
            .groupBy("name")\
                .agg(
                    sum(col("volume")).alias("total_volume")
                )\
            .orderBy(
                col("total_volume").desc())\
    .show()

+-----------+------------+
|       name|total_volume|
+-----------+------------+
|Central Hub|     4200000|
|North Depot|     2448000|
| South Wing|     1800000|
+-----------+------------+

